# Colab: signer-independent ISL landmark model experiments

**Training is blocked until you provide real, licensed ISL training clips with documented signer IDs.** This notebook never creates synthetic training data or synthetic metric rows. Do not upload WLASL videos for an ISL claim (WLASL is ASL). Use pseudonymous signer IDs and ensure the people shown consented to recording and cloud processing. Review `data/sources.md` and verify every dictionary sign with a fluent ISL signer.

The Drive input below must be the licensed external training dataset, not `data/raw/own/` recordings. Keep any clips recorded with `tools.record_clips` in the own-recording folder as a separate held-out live-demo set; never copy them into the training manifest.

The default experiment grid compares BiLSTM, BiGRU, Transformer, and TCN across hands-only vs. hands+pose, augmentation on/off, three sequence lengths, and three seeds. This is a long run; save the Drive folder and keep the emitted MLflow and per-run JSON records.

In [ ]:
# Select one: clone a pushed repository or upload/expand your repository ZIP.
REPO_URL = "REPLACE_WITH_YOUR_GIT_REPOSITORY_URL"
assert REPO_URL.startswith("https://"), "Set REPO_URL to your HTTPS repository URL first."
!git clone {REPO_URL} /content/isl-project
%cd /content/isl-project
!git rev-parse --short HEAD

In [ ]:
!python -m pip install --upgrade pip
!python -m pip install -e ".[train,app,dev]"

In [ ]:
from pathlib import Path

from google.colab import drive

drive.mount('/content/drive')
DATA_DIR = Path('/content/drive/MyDrive/isl-licensed-training-clips')
assert (DATA_DIR / 'clips.csv').is_file(), (
    'Place the licensed external training clips.csv here. It must contain video_path,label,signer_id; do not use data/raw/own/clips.csv.'
)
# Copy media into the runtime. Keep it in private Drive storage and delete the
# Colab runtime afterwards if your consent/data policy requires it.
!mkdir -p data/raw/downloaded/videos
!cp "{DATA_DIR}/clips.csv" data/raw/downloaded/clips.csv
!cp -r "{DATA_DIR}/videos/." data/raw/downloaded/videos/

In [ ]:
# Validate video readability, signer IDs and signer-independent splits first.
!python -m tools.make_manifest --clips data/raw/downloaded/clips.csv --output data/processed/clips_manifest.csv --report reports/data_manifest.json
# Extract every available clip. Inspect preprocessing_errors.txt if emitted.
!python -m src.isl_recognition.data --source-csv data/processed/clips_manifest.csv --output-dir data/processed

In [ ]:
import json
from pathlib import Path

manifest = Path('data/processed/manifest.csv')
assert manifest.is_file(), 'Feature extraction did not produce a manifest.'
with open('reports/data_manifest.json', encoding='utf-8') as f:
    print(json.dumps(json.load(f), indent=2))
print('Feature manifest rows:', sum(1 for _ in manifest.open(encoding='utf-8')) - 1)
assert sum(1 for _ in manifest.open(encoding='utf-8')) > 1, 'No extracted clips; stop without training.'

In [ ]:
# This grid is intentionally exhaustive and can take hours. No GPU is required,
# but a GPU runtime is strongly recommended. Checkpointing selects on validation
# metrics; held-out test data is used only for the final evaluation.
!python -m src.isl_recognition.train_landmarks --manifest data/processed/manifest.csv --architectures lstm gru transformer tcn --seeds 42 43 44 --modalities hands hands_pose --augmentations on off --sequence-lengths 16 32 48 --epochs 40 --batch-size 32

In [ ]:
# Preserve checkpoints, MLflow output, exact run logs, and reports in Drive.
from datetime import datetime, timezone

destination = Path('/content/drive/MyDrive/isl-model-runs') / datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
destination.mkdir(parents=True, exist_ok=False)
!cp -r models "{destination}/models"
!cp -r reports "{destination}/reports"
!cp -r mlruns "{destination}/mlruns"
print('Saved experiment artifacts to', destination)